# 01: Community-Driven Data Collection for African Oral Traditions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muchiriTimdev/ngumzo.ai/blob/main/notebooks/01_data_collection.ipynb)

This notebook provides tools and protocols for ethically collecting oral tradition data from African communities.

## Goals
- Establish ethical data collection protocols
- Create structured metadata schemas
- Build recording and transcription workflows
- Ensure community consent and attribution

## Prerequisites
- Understanding of ethical research practices
- Community relationships and permissions
- Recording equipment (smartphone minimum, professional mic recommended)

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q pydub librosa soundfile pandas numpy requests gradio

# For Colab: Mount Google Drive for data storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import uuid

# Audio processing
import librosa
import soundfile as sf
from pydub import AudioSegment

# Create project directories
DATA_ROOT = Path('/content/drive/MyDrive/african_oral_llm_data')  # Change for local
RAW_AUDIO_DIR = DATA_ROOT / 'raw_audio'
PROCESSED_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
TRANSCRIPTS_DIR = DATA_ROOT / 'transcripts'

for dir_path in [RAW_AUDIO_DIR, PROCESSED_DIR, METADATA_DIR, TRANSCRIPTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"✓ Data directories created at: {DATA_ROOT}")

## 2. Ethical Framework and Consent Management

In [ ]:
class ConsentManager:
    """
    Manages informed consent for oral tradition data collection.
    Ensures ethical compliance and data sovereignty.
    """
    
    CONSENT_LEVELS = {
        'research_only': 'Data used for academic research only',
        'open_access': 'Data shared openly with attribution',
        'community_control': 'Community retains control of usage',
        'commercial_excluded': 'Non-commercial use only'
    }
    
    def __init__(self, metadata_dir):
        self.consent_db_path = Path(metadata_dir) / 'consent_records.json'
        self.consent_records = self._load_consent_db()
    
    def _load_consent_db(self):
        if self.consent_db_path.exists():
            with open(self.consent_db_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {}
    
    def record_consent(self, contributor_id, contributor_name, consent_level, 
                      language_community, notes="", audio_signature=None):
        """
        Record informed consent from a contributor.
        
        Args:
            contributor_id: Unique identifier for contributor
            contributor_name: Name (can be pseudonym)
            consent_level: One of CONSENT_LEVELS keys
            language_community: e.g., 'Kikongo', 'Yoruba', etc.
            notes: Additional context
            audio_signature: Path to audio recording of verbal consent
        """
        if consent_level not in self.CONSENT_LEVELS:
            raise ValueError(f"Invalid consent level. Choose from: {list(self.CONSENT_LEVELS.keys())}")
        
        record = {
            'contributor_id': contributor_id,
            'contributor_name': contributor_name,
            'consent_level': consent_level,
            'consent_description': self.CONSENT_LEVELS[consent_level],
            'language_community': language_community,
            'date_recorded': datetime.now().isoformat(),
            'notes': notes,
            'audio_signature': audio_signature,
            'data_items': []
        }
        
        self.consent_records[contributor_id] = record
        self._save_consent_db()
        
        return record
    
    def link_data_to_consent(self, contributor_id, data_item_id):
        """Link a data item to a contributor's consent record."""
        if contributor_id in self.consent_records:
            self.consent_records[contributor_id]['data_items'].append(data_item_id)
            self._save_consent_db()
    
    def _save_consent_db(self):
        with open(self.consent_db_path, 'w', encoding='utf-8') as f:
            json.dump(self.consent_records, f, indent=2, ensure_ascii=False)
    
    def get_consent_record(self, contributor_id):
        return self.consent_records.get(contributor_id)
    
    def can_use_for_training(self, contributor_id):
        """Check if data can be used for model training."""
        record = self.consent_records.get(contributor_id)
        if not record:
            return False
        return record['consent_level'] in ['research_only', 'open_access', 'community_control']

# Initialize consent manager
consent_manager = ConsentManager(METADATA_DIR)
print("✓ Consent Manager initialized")
print(f"  Consent levels: {list(ConsentManager.CONSENT_LEVELS.keys())}")

### Example: Recording Consent

In [ ]:
# Example: Record consent for a contributor (run this for each contributor)
contributor = consent_manager.record_consent(
    contributor_id=str(uuid.uuid4()),
    contributor_name="Elder Nkosi (pseudonym)",
    consent_level="community_control",
    language_community="Kikongo",
    notes="Elder from Mbanza Kongo, specializes in oral histories and proverbs",
    audio_signature="consent_recordings/nkosi_consent.wav"
)

print(f"✓ Consent recorded for: {contributor['contributor_name']}")
print(json.dumps(contributor, indent=2, ensure_ascii=False))

## 3. Metadata Schema for Oral Traditions

In [ ]:
class OralTraditionMetadata:
    """
    Structured metadata for African oral tradition recordings.
    Captures cultural context, performance details, and linguistic features.
    """
    
    GENRES = [
        'proverb', 'folk_tale', 'epic_poetry', 'lullaby', 'work_song',
        'praise_poetry', 'divination', 'oral_history', 'myth', 'legend',
        'riddle', 'tongue_twister', 'prayer', 'incantation'
    ]
    
    PERFORMANCE_CONTEXTS = [
        'ceremony', 'marketplace', 'home_evening', 'initiation', 'funeral',
        'harvest_festival', 'wedding', 'conflict_resolution', 'healing',
        'storytelling_competition', 'everyday_conversation'
    ]
    
    @staticmethod
    def create_metadata(
        recording_id,
        contributor_id,
        title,
        language_code,  # e.g., 'kik', 'yor', 'swa'
        dialect_region,
        genre,
        performance_context,
        duration_seconds,
        collector_notes="",
        cultural_significance="",
        age_estimated=None,
        gender="",
        is_tonal_language=True,
        translation_notes="",
        related_stories=None,
        keywords=None,
        transcription_status='pending'
    ):
        """
        Create a comprehensive metadata record.
        
        Args:
            recording_id: Unique identifier
            contributor_id: Links to consent record
            title: Story/song title in original language
            language_code: ISO 639-3 code
            dialect_region: Specific dialect/region
            genre: Type of oral tradition
            performance_context: Occasion of recording
            duration_seconds: Audio length
            collector_notes: Researcher observations
            cultural_significance: Community-provided context
            age_estimated: Performer age estimate
            gender: Performer gender
            is_tonal_language: Whether language is tonal
            translation_notes: Translation methodology notes
            related_stories: Linked narrative IDs
            keywords: Searchable tags
            transcription_status: 'pending', 'in_progress', 'complete'
        """
        
        return {
            'recording_id': recording_id,
            'contributor_id': contributor_id,
            'title': {
                'original': title,
                'transliteration': "",  # To be filled
                'translation_english': ""
            },
            'language': {
                'code': language_code,
                'name_native': "",
                'name_english': "",
                'dialect_region': dialect_region,
                'is_tonal': is_tonal_language,
                'writing_system': 'mandombe' if language_code == 'kik' else 'latin'
            },
            'content': {
                'genre': genre,
                'performance_context': performance_context,
                'cultural_significance': cultural_significance,
                'abstract_summary': "",
                'keywords': keywords or [],
                'related_stories': related_stories or []
            },
            'recording': {
                'duration_seconds': duration_seconds,
                'date_recorded': datetime.now().isoformat(),
                'location': {
                    'village': "",
                    'region': "",
                    'country': "",
                    'gps_coordinates': None  # If permitted
                },
                'equipment': "",
                'audio_quality': "",  # 'excellent', 'good', 'fair', 'poor'
                'ambient_noise': "",
                'number_of_speakers': 1
            },
            'performer': {
                'age_estimated': age_estimated,
                'gender': gender,
                'social_role': "",  # e.g., 'griot', 'elder', 'mother', 'chief'
                'performance_history': ""
            },
            'documentation': {
                'collector_notes': collector_notes,
                'translation_notes': translation_notes,
                'transcription_status': transcription_status,
                'reviewed_by_community': False
            },
            'technical': {
                'sample_rate': 48000,  # Recommended minimum
                'bit_depth': 24,
                'channels': 1,
                'file_format': 'wav',
                'raw_file_path': "",
                'processed_file_path': ""
            },
            'rights': {
                'consent_level': consent_manager.get_consent_record(contributor_id)['consent_level'],
                'attribution_required': True,
                'commercial_use_permitted': False,
                'community_ownership': True
            }
        }

print("✓ Metadata schema defined")
print(f"  Available genres: {OralTraditionMetadata.GENRES}")
print(f"  Performance contexts: {OralTraditionMetadata.PERFORMANCE_CONTEXTS}")

## 4. Audio Recording and Upload Interface

In [ ]:
# Gradio interface for recording/uploading audio with metadata

import gradio as gr

def process_audio_upload(audio_file, contributor_id, title, language_code, 
                        dialect_region, genre, context, collector_notes,
                        cultural_notes, age, gender):
    """Process uploaded/recorded audio and save with metadata."""
    
    if audio_file is None:
        return "Error: No audio file provided"
    
    # Generate unique ID
    recording_id = str(uuid.uuid4())
    
    # Load audio to get duration
    try:
        y, sr = librosa.load(audio_file, sr=None)
        duration = len(y) / sr
        
        # Save raw audio
        raw_path = RAW_AUDIO_DIR / f"{recording_id}.wav"
        sf.write(raw_path, y, sr, subtype='PCM_24')
    except Exception as e:
        return f"Error processing audio: {str(e)}"
    
    # Create metadata
    metadata = OralTraditionMetadata.create_metadata(
        recording_id=recording_id,
        contributor_id=contributor_id,
        title=title,
        language_code=language_code,
        dialect_region=dialect_region,
        genre=genre,
        performance_context=context,
        duration_seconds=duration,
        collector_notes=collector_notes,
        cultural_significance=cultural_notes,
        age_estimated=age if age else None,
        gender=gender,
        is_tonal_language=True
    )
    
    # Update file paths
    metadata['technical']['raw_file_path'] = str(raw_path)
    
    # Save metadata
    meta_path = METADATA_DIR / f"{recording_id}.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    # Link to consent
    consent_manager.link_data_to_consent(contributor_id, recording_id)
    
    return f"✓ Recording saved!\nID: {recording_id}\nDuration: {duration:.1f}s\nSaved to: {raw_path}"

# Build Gradio interface
with gr.Blocks(title="African Oral Tradition Data Collection") as demo:
    gr.Markdown("""
    # 🎙️ African Oral Tradition Data Collection
    
    Record or upload audio with complete metadata for ethical AI training data.
    
    **Important**: Ensure you have explicit consent from the contributor before recording.
    """)
    
    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(
                label="Record or Upload Audio",
                source="microphone",
                type="filepath"
            )
            
            contributor_id = gr.Textbox(
                label="Contributor ID",
                placeholder="From consent registration"
            )
            
            title = gr.Textbox(
                label="Title (in original language)",
                placeholder="e.g., Kinkulu kia nzambi"
            )
            
            language_code = gr.Dropdown(
                choices=['kik', 'yor', 'swa', 'zul', 'xho', 'wol', 'aka'],
                label="Language Code (ISO 639-3)"
            )
            
            dialect_region = gr.Textbox(
                label="Dialect/Region",
                placeholder="e.g., Kikongo ya L'Etat, Mbanza Kongo"
            )
        
        with gr.Column():
            genre = gr.Dropdown(
                choices=OralTraditionMetadata.GENRES,
                label="Genre/Type"
            )
            
            context = gr.Dropdown(
                choices=OralTraditionMetadata.PERFORMANCE_CONTEXTS,
                label="Performance Context"
            )
            
            collector_notes = gr.TextArea(
                label="Collector Notes",
                placeholder="Observations about the recording, performance style, etc."
            )
            
            cultural_notes = gr.TextArea(
                label="Cultural Significance",
                placeholder="Context provided by community/elder"
            )
            
            with gr.Row():
                age = gr.Number(label="Age (est.)", minimum=0, maximum=120)
                gender = gr.Dropdown(
                    choices=['male', 'female', 'other', ''],
                    label="Gender"
                )
    
    submit_btn = gr.Button("Save Recording & Metadata", variant="primary")
    output = gr.Textbox(label="Status", lines=5)
    
    submit_btn.click(
        fn=process_audio_upload,
        inputs=[
            audio_input, contributor_id, title, language_code,
            dialect_region, genre, context, collector_notes,
            cultural_notes, age, gender
        ],
        outputs=output
    )

demo.launch(debug=True)

## 5. Batch Processing Existing Audio Files

In [ ]:
def batch_process_audio_files(audio_paths, contributor_id, common_metadata):
    """
    Process multiple audio files with shared metadata.
    
    Args:
        audio_paths: List of paths to audio files
        contributor_id: Consent ID for all files
        common_metadata: Dict with shared metadata fields
    """
    results = []
    
    for i, audio_path in enumerate(audio_paths):
        try:
            # Generate ID
            recording_id = str(uuid.uuid4())
            
            # Load and validate
            y, sr = librosa.load(audio_path, sr=48000)
            duration = len(y) / sr
            
            # Save standardized file
            raw_path = RAW_AUDIO_DIR / f"{recording_id}.wav"
            sf.write(raw_path, y, 48000, subtype='PCM_24')
            
            # Create metadata
            metadata = OralTraditionMetadata.create_metadata(
                recording_id=recording_id,
                contributor_id=contributor_id,
                title=common_metadata.get('title_prefix', '') + f"_{i+1}",
                language_code=common_metadata['language_code'],
                dialect_region=common_metadata.get('dialect_region', ''),
                genre=common_metadata.get('genre', 'oral_history'),
                performance_context=common_metadata.get('context', 'home_evening'),
                duration_seconds=duration
            )
            
            metadata['technical']['raw_file_path'] = str(raw_path)
            
            # Save
            meta_path = METADATA_DIR / f"{recording_id}.json"
            with open(meta_path, 'w', encoding='utf-8') as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)
            
            consent_manager.link_data_to_consent(contributor_id, recording_id)
            
            results.append({'id': recording_id, 'status': 'success'})
            
        except Exception as e:
            results.append({'path': audio_path, 'status': 'error', 'error': str(e)})
    
    return results

# Example usage:
# audio_files = ['/path/to/file1.wav', '/path/to/file2.wav']
# common_meta = {'language_code': 'kik', 'genre': 'proverb', 'title_prefix': 'proverbs_session'}
# results = batch_process_audio_files(audio_files, 'contributor-uuid', common_meta)

## 6. Quality Control and Validation

In [ ]:
class AudioValidator:
    """Validates audio quality for AI training suitability."""
    
    @staticmethod
    def validate_audio(audio_path, min_duration=5, max_duration=600):
        """
        Check if audio meets quality standards.
        
        Returns dict with validation results and recommendations.
        """
        issues = []
        warnings = []
        
        try:
            # Load audio
            y, sr = librosa.load(audio_path, sr=None)
            duration = len(y) / sr
            
            # Check duration
            if duration < min_duration:
                issues.append(f"Duration too short: {duration:.1f}s < {min_duration}s")
            if duration > max_duration:
                warnings.append(f"Duration long: {duration:.1f}s - consider segmenting")
            
            # Check sample rate
            if sr < 16000:
                issues.append(f"Sample rate too low: {sr}Hz < 16000Hz")
            
            # Check signal levels
            rms = np.sqrt(np.mean(y**2))
            if rms < 0.01:
                issues.append("Audio level too low - may be inaudible")
            if rms > 0.9:
                warnings.append("Audio may be clipped/distorted")
            
            # Check for silence (more than 2s of near-silence)
            hop_length = int(sr * 0.01)  # 10ms hop
            rms_frames = librosa.feature.rms(y=y, hop_length=hop_length)[0]
            silence_threshold = 0.005
            long_silence_frames = int(2.0 * sr / hop_length)  # 2 seconds
            
            consecutive_silence = 0
            max_silence = 0
            for rms_val in rms_frames:
                if rms_val < silence_threshold:
                    consecutive_silence += 1
                    max_silence = max(max_silence, consecutive_silence)
                else:
                    consecutive_silence = 0
            
            if max_silence > long_silence_frames:
                warnings.append("Long periods of silence detected")
            
            # Estimate SNR (simplified)
            signal_power = np.mean(y**2)
            
            return {
                'valid': len(issues) == 0,
                'issues': issues,
                'warnings': warnings,
                'duration': duration,
                'sample_rate': sr,
                'rms_level': float(rms),
                'max_silence_sec': max_silence * hop_length / sr
            }
            
        except Exception as e:
            return {
                'valid': False,
                'issues': [f"Error processing: {str(e)}"],
                'warnings': []
            }

# Example validation
# validator = AudioValidator()
# result = validator.validate_audio('/path/to/recording.wav')
# print(result)

## 7. Export Dataset Summary

In [ ]:
def generate_dataset_summary():
    """Generate summary statistics of collected data."""
    
    # Load all metadata
    all_metadata = []
    for meta_file in METADATA_DIR.glob('*.json'):
        with open(meta_file, 'r', encoding='utf-8') as f:
            all_metadata.append(json.load(f))
    
    if not all_metadata:
        return "No data collected yet."
    
    # Calculate statistics
    df = pd.json_normalize(all_metadata)
    
    summary = {
        'total_recordings': len(all_metadata),
        'total_duration_hours': sum(m['recording']['duration_seconds'] for m in all_metadata) / 3600,
        'languages': list(set(m['language']['code'] for m in all_metadata)),
        'genres': {},
        'contexts': {},
        'transcription_status': {}
    }
    
    # Count genres
    for m in all_metadata:
        genre = m['content']['genre']
        summary['genres'][genre] = summary['genres'].get(genre, 0) + 1
        
        context = m['content']['performance_context']
        summary['contexts'][context] = summary['contexts'].get(context, 0) + 1
        
        status = m['documentation']['transcription_status']
        summary['transcription_status'][status] = summary['transcription_status'].get(status, 0) + 1
    
    return summary

# Generate and display summary
summary = generate_dataset_summary()
print("📊 Dataset Summary")
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Next Steps

1. **Proceed to Notebook 02**: Process collected audio for tonal analysis
2. **Export metadata**: Use `generate_dataset_summary()` to track progress
3. **Begin transcription**: Manual or semi-automated transcription workflows

## References

- [UNESCO Intangible Cultural Heritage](https://ich.unesco.org/)
- [FAIR Principles for Indigenous Data](https://www.gida-global.org/sowhat)
- [Responsible AI for Digital Heritage](https://core.ac.uk/download/pdf/286050010.pdf)